# CRS Inventory – Boundary Layers

This notebook inspects the coordinate reference systems (CRS) of boundary layers:

- County boundary
- City boundary
- TOC boundaries
- Village boundaries

**Goals:**
- Load each boundary dataset.
- Print its CRS in a readable format.
- Identify mismatches and potential issues.
- Start to narrow down a suitable **project CRS**.

In [1]:
import os
from pathlib import Path

import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"

BOUNDARIES_DIR

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries')

## Boundary layers included

The following boundary datasets are expected in `data/boundaries/`:

- `City_of_Phoenix_City_Limits_Boundary.shp` – City of Phoenix corporate boundary.
- `US_county_2023.shp` - US counties, which will later be clipped to just Maricopa County.
- `Walkable_Urban_Code.shp` – Transit-Oriented Communities (TOC) boundaries.
- `Villages.shp` – Village boundaries for Phoenix.

The next cell defines a dictionary mapping descriptive names to their file paths.

In [9]:
boundary_files = {
    "phoenix_city_boundary": BOUNDARIES_DIR / "city/City_of_Phoenix_City_Limits_Boundary.shp",
    "phoenix_county_boundary": BOUNDARIES_DIR / "county/US_county_2023.shp",
    "phoenix_toc_boundaries": BOUNDARIES_DIR / "tocs/Walkable_Urban_Code.shp",
    "phoenix_village_boundaries": BOUNDARIES_DIR / "villages/Villages.shp",
}

boundary_files

{'phoenix_city_boundary': WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/city/City_of_Phoenix_City_Limits_Boundary.shp'),
 'phoenix_county_boundary': WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/county/US_county_2023.shp'),
 'phoenix_toc_boundaries': WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/tocs/Walkable_Urban_Code.shp'),
 'phoenix_village_boundaries': WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/villages/Villages.shp')}

In [10]:
def inspect_crs(layer_name: str, path: Path):
    """Load a shapefile into a GeoDataFrame and print its CRS.

    Parameters
    ----------
    layer_name : str
        Short descriptive name of the layer.
    path : Path
        Full path to the shapefile.
    """
    if not path.exists():
        print(f"[MISSING] {layer_name}: {path}")
        return

    try:
        gdf = gpd.read_file(path)
        print(f"\n=== {layer_name} ===")
        print(f"Path: {path}")
        print(f"CRS: {gdf.crs}")
        print(f"Number of features: {len(gdf)}")
    except Exception as e:
        print(f"[ERROR] {layer_name}: {e}")


In [11]:
for name, path in boundary_files.items():
    inspect_crs(name, path)


=== phoenix_city_boundary ===
Path: c:\Users\arjav\DevSource\toc-performance-dashboard\data\boundaries\city\City_of_Phoenix_City_Limits_Boundary.shp
CRS: EPSG:2868
Number of features: 1

=== phoenix_county_boundary ===
Path: c:\Users\arjav\DevSource\toc-performance-dashboard\data\boundaries\county\US_county_2023.shp
CRS: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",37.5],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102

## CRS notes

All local boundary layers use the same CRS - EPSG:2868, which is the NAD83 Arizona Central projection. The county shapefile is for the entire nation, and that uses the Albers Equal Area Conic projection.

We will later decide on a single `PROJECT_CRS` to use for all analysis. That decision will also consider the CRS used by parcel, census, and LODES layers.